### THE BIG PICTURE: Part II at a Glance
   Part II is a completely different from Part I. Part I was about building a
   processor from gates up (RISC-V, datapath, control, pipelining). Part II 
   flips the perspective -- you're now the programmer looking down at the machine
   . The ISA switches from RISC-V to x86-64 (a CISC architecture), the syntax
   is AT&T (source, destination -- opposite to what you're used to from RISC-V!)
   , ...


### Block A -- "How does C become machine code?"
   This is the core assembly programming block, building up from simple to 
   complex:

   LECTURE 1 -- The Foundation. x86-64 registers (16 general-purpose, with 
   32/16/8-bit sub-registers), operand types (immediate `$`, register `%`,
   memory `()`), the addressing mode formula 
   `IMM(rb, ri, s) = Mem[R[rb] + s*R[ri] + Imm]`, and `mov` instructions.
   Also: ISA vs microarchitecture, CISC vs RISC, x86 history. This is the 
   vocabulary -- every later lecture assumes you can read addressing modes
   fluently.

   LECTURE 2 -- Arithmetic + Control Flow. Two-operand arithmetic (`addl`, 
   `subl`, `imull`, shifts), one-operand ops (`incl`, `negl`), and the crucial
   `lea` isntruction (address math without memory access -- the compiler's
   favourite trick for fast arithmetic). Then: condition codes (CF, ZF, SF, OF),
   how they're set implicitly by arithmetic or explicitly by `cmp`/`test`, 
   `set*` instructions to read flags, `j*` for conditional jumps, and `cmov*`
   for branchless conditionals.

   LECTURE 3 -- Loops + Switch. How the Compiler translates `do-while`, `while`
   (jump-to-middle vs guarded-do), and `for` loops into assembly using 
   conditional jumps. Then switch statements: compact cases use a JUMP TABLE
   (array of code addreses, indexed by the case value, O(1) lookup), sparse
   cases use a BINARY DECISION TREE (O(log n) comparisons).

   LECTURE 4 -- Stack + Procedures. The runtime stack, `call`/`ret` mechanics
   (push return address, pop and jump back), stack frames, the x86-64 calling
   convention (first 6 integer args in `%rdi, %rsi, %rdx, %rcx, %r8, %r9`, 
   return in `%rax`, extras on the stack), caller-saved vs callee-saved 
   registers, and how the compiler decides when a function even needs a stack
   frame at all.

   LECTURE 5 -- Arrays + Structs. How 1D arrays become base+index*scale 
   addressing, nested vs multi-level 2D arrays (contiguous block vs array of
   pointers), struct layoput in memory, and ALIGNMENT (K-byte types must sit at
   addresses that are multiples of K, leading to padding bytes -- internal
   fragmentation).


### Block B -- "Why is my program slow?" (Lectures 6-7)
   Lecture 6 -- Memory Hierarchy. SRAM vs DRAM, the pyramid (registers -> L1
   -> L2 -> L3 -> DRAM -> DISK), why it works (temporal + spatial locality), 
   cache basics (hit/miss, placement, replacement), direct-mapped caches, and 
   the AMAT formula: `AMAT = Hit Time + Miss Rate \times Miss Penalty`. The
   row-major vs column-major array access example is a classic example question
   -- stride-1 access gives spatial locality, stride-N access destroys it.

   Lecture 7 -- Caches in Depth. The address decomposition. 
   `[Tag | Index | Offset]` with `I = log_2 (C/B/N) and `O = log_2 (B)`. 
   Direct-mapped (N=1), set-associative (N = 2, 4, 8...), fully assosciative 
   (N=C/B). Repalcement policies (LRU). Write policies (write-back vs 
   write-through, write-allocate vs no-write-allocate). Real cache specs
   ... Software optimisations: loop reordering for locality, blocking/tiling.

---

### Lecture 1+2: x86-64 Registers, Addressing, Arithmetic & Control Flow

The big mental shift from Part I
   In Part I you worked with RISC-V -- a clean, regular ISA where every 
   instruction is 32 bits, there are 32 registers named `x0`-`x31`, and the
   syntax is `op rd, rs1, rs2` (destination first). You also built the hardware
   that executes these instructions. 

   x86-64 is the opposite of clean. It's a CISC architecture that has been 
   accumulating features since 1978. Instructions can be 1 to 15 bytes long. 
   There are only 16 general-purpose registers with weird historical names. The
   AT&T syntax puts the SOURCE FIRST, DESTINATION SECOND -- backwards from
   RISC-Vs. And arithmetic instructions can operate directly on memory, not 
   just registers.

   But here's the connection to Part 1 that makes it all click: the ISA is just
   the contract between the programmer and the hardware. In Part I, you built
   the hardware side of that contract for RISC-V. Now you're the programmer side
   on a different contract -- x86-64, The concepts (registers, ALU operations,
   memory addressing, branches) are all the same, only the syntax and encoding
   details change.


---
The 16 Registers
   x86-64 has 16 general-purpose 64-bit integer registers. The first 8 have
   historical names from the 1-bit 8086 era; the last 8 were added when AMD 
   extended x86 to 64 bits:

   `%rax` (return value), `%rbx`, `%rcx`, `%rdx`, `%rsi`, `%rdi`, `%rsp` (stack
   pointer -- don't touch), `%rbp` (frame pointer -- optional in x8-64), `%r8`
   through `%r15`.

   The crucial thing is that each register has SUB-REGISTER names that access 
   smaller portions of the same physical register. For example, `%rax` is the 
   full 64 bits, `%eax` is the bottom 32 bits, `%ax` is the bottom 16, `%al` is
   the bottom 8. When you write to `%eax`, the top 32 bits of `%rax` get zeroed.
   When you write to `%ax` or `%al`, the upper bits are left unchanged. This
   matters because the compiler picks the smallest suffix it can get away with
   -- if your C code uses `int` (32 bits), you'll see `%eax`, `%edx` etc., not
   the 64-bit names.

   The other key fact: THE CALLING CONVENTION ASSIGNS SPECIFIC ROLES to these 
   registers. The first six integer arguments to a function arrive in 
   `%rdi, %rsi, %rdx, %rcx, %r8, %r9` (in that order). The return value goes in
   `%rax`. So when you see `movl %edi, %eax` at the start of a function, that's 
   just "copy the first argument into the register I'll use for
    computation/return".

---